# **Civil and Penal Court Data Processing: Summary and Report**


### This notebook provides an overview of the processing and indexing of court decision data into Elasticsearch from both civil and penal court datasets. Key points include:

1. **Civil Court Data**: 
   - **Files**: `decis_cjc_acjc.json` and `decis_cjc_sommaires.json`.
   - **Stats**: 7031 total records in the `acjc` dataset and 4182 in the `sommaires` dataset.
   - **Issues Identified**: 1 missing record and 32 duplicate records in the `sommaires` dataset. No missing records in `acjc`, but there was a mismatch of 1 document.

2. **Penal Court Data**:
   - **Files**: Two datasets, `df_parp` (3402 rows) and `df_pcpr` (4981 rows), totaling 8383 records.
   - **Issues Identified**: 1 missing row and 146 rows with duplicate IDs across datasets.

### Latest Updates:

In this notebook, we use **`cle_fiche`** as the unique index for ensuring that IDs are consistently unique across all records. We validate that **no record contains null or empty values** for this field, which is critical to maintaining the integrity and reliability of the data before loading it into Elasticsearch.

In [21]:
from __future__ import annotations

import json
import logging

from datetime import datetime
from pathlib import Path
from typing import Any, Generator

import pandas as pd
from elasticsearch import Elasticsearch

pd.set_option('display.max_rows', 10)



In [22]:
class DataReader:
    """Helper class for reading large data from files."""

    SUPPORTED_EXTENSIONS = ["jsonl", "json"]

    def __init__(self, file_path: str):
        if not Path(file_path).is_file():
            raise FileNotFoundError(f"File {file_path} does not exist")
        self.file_path = Path(file_path)

    def _read_json_lines(
        self,
    ) -> Generator[tuple[int, dict[str, Any]], None, None]:
        with open(self.file_path, "r", encoding="utf-8") as f:
            for line_number, line in enumerate(f):
                try:
                    json_obj = json.loads(line.strip())  # remove \n
                    yield line_number, json_obj
                except Exception as e:
                    raise e

    def read_data(self) -> Generator[tuple[int, dict[str, Any]], None, None]:
        if self.file_path.suffix == ".jsonl" or "json":
            return self._read_json_lines()
        else:
            raise NotImplementedError(
                f"DataReader doesn't support {self.file_path.suffix} file extension. "
                f"Supported extensions: {self.SUPPORTED_EXTENSIONS}"
            )


In [131]:
from typing import Set

class DataReader:
    """Helper class for reading large data from files."""

    SUPPORTED_EXTENSIONS = ["jsonl"]

    def __init__(self, file_path: str):
        if not Path(file_path).is_file():
            raise FileNotFoundError(f"File {file_path} does not exist")
        self.file_path = Path(file_path)

    def _read_jsonl_lines(
        self,
    ) -> Generator[tuple[int, dict[str, Any]], None, None]:
        with open(self.file_path, "r", encoding="utf-8") as f:
            for line_number, line in enumerate(f):
                try:
                    json_obj = json.loads(line.strip())  # remove \n
                    yield line_number, json_obj
                except Exception as e:
                    raise e

    def read_data(self) -> Generator[tuple[int, dict[str, Any]], None, None]:
        if self.file_path.suffix == ".json" or ".jsonl":
            return self._read_jsonl_lines()
        else:
            raise NotImplementedError(
                f"DataReader doesn't support {self.file_path.suffix} file extension. "
                f"Supported extensions: {self.SUPPORTED_EXTENSIONS}"
            )

    def count_items(self) -> int:
        """
        Count the number of items in the JSONL file.

        Returns
        -------
        int
            The number of items (lines) in the file.
        """
        return sum(1 for _ in self.read_data())

    def collect_keys(self) -> Set[str]:
        """
        Collect unique keys from all the items in the JSONL file.

        Returns
        -------
        Set[str]
            A set of all unique keys in the JSON objects.
        """
        all_keys = set()
        for _, document in self.read_data():
            all_keys.update(document.keys())
        return all_keys

    def retrieve_document_fields(self):
        """
        Reads data and retrieves specific fields from each document.
    
        Returns
        -------
        list
            A list of dictionaries, where each dictionary contains the fields
            'document_id', 'document_datedecision', and 'document_dt_decision' for a document.
        """
        dict_ = []
        #  case_nature, case_type, or collector_name
        for _, document in self.read_data():
            doc_dict = {
                "document_id": document.get("id"),
                "dt_decision": document.get("dt_decision"),
                "document_procedure": document.get("procedure"),
                "nature": document.get("nature"),
                "type": document.get("type"),
                "coll_nom": document.get("coll_nom"),
            }
            # Append the document dictionary to the list
            dict_.append(doc_dict)
        return pd.DataFrame(dict_)


### PENAL COURT

In [153]:
dataset = {
    "parp": "Penal_Court/decis_parp.json", 
    "pcpr": "Penal_Court/decis_pcpr.json"
}

In [154]:
for data, path in dataset.items():
    if data == "parp":
        reader = DataReader(file_path=path)
        df_parp = reader.retrieve_document_fields()
        print(reader.collect_keys())
    if data == "pcpr":
        reader = DataReader(file_path=path)
        df_pcpr = reader.retrieve_document_fields()
        print(reader.collect_keys())

{'source', 'n_ext_proc', 'coll_nom', 'datedecision', 'n_ext_jur_attr', 'document_text', 'cle_fiche', 'decision', 'relations', 'parties', 'resultat', 'recours', 'publieinternet', 'query', 'nature', 'resume', 'importance', 'descripteurs', 'fichierword', 'document', 'rectification', 'normes', 'dt_decision', 'arretdeprincipe', 'proc_year', 'id', 'procedure'}
{'source', 'n_ext_proc', 'coll_nom', 'datedecision', 'n_ext_jur_attr', 'document_text', 'cle_fiche', 'decision', 'relations', 'parties', 'resultat', 'recours', 'publieinternet', 'query', 'nature', 'resume', 'importance', 'fichierword', 'descripteurs', 'document', 'rectification', 'normes', 'dt_decision', 'proc_year', 'arretdeprincipe', 'id', 'procedure'}


In [155]:
df_parp

,document_id,dt_decision,document_procedure,nature,type,coll_nom
0,1674728215,26.01.2023,P/13187/2020,PENAL,None,parp
1,1650973217,07.04.2022,P/1544/2017,PENAL,None,parp
2,1669304425,08.11.2022,P/7877/2020,PENAL,None,parp
3,1663331640,12.08.2022,P/18166/2019,PENAL,None,parp
4,1675851771,08.02.2023,P/24065/2021,PENAL,None,parp
...,...,...,...,...,...,...
3397,1725868998,05.09.2024,P/20629/2023,PENAL,None,parp
3398,1717749308,23.05.2024,P/22394/2014,PENAL,None,parp
3399,1725958327,11.07.2024,P/10477/2020,PENAL,None,parp
3400,1725608637,30.08.2024,P/7282/2022,PENAL,None,parp


In [156]:
df_pcpr

,document_id,dt_decision,document_procedure,nature,type,coll_nom
0,1686558105,09.06.2023,P/14428/2021,MP,None,pcpr
1,1686667832,12.06.2023,P/4105/2023,MP,None,pcpr
2,1686668437,12.06.2023,P/18999/2022,MP,None,pcpr
3,1685017202,24.05.2023,P/19808/2019,MP,None,pcpr
4,1686755369,13.06.2023,P/19549/2019,MP,None,pcpr
...,...,...,...,...,...,...
4976,1720695305,10.07.2024,P/2584/2023,JMI,None,pcpr
4977,1669044099,17.11.2022,P/1313/2019,MP,None,pcpr
4978,1720769558,11.07.2024,P/8138/2018,MP,None,pcpr
4979,1702540901,13.12.2023,P/6598/2023,MP,None,pcpr


In [134]:
# Convert dt_decision to datetime
df_parp['dt_decision'] = pd.to_datetime(df_parp['dt_decision'], format='%d.%m.%Y', errors='coerce')
df_pcpr['dt_decision'] = pd.to_datetime(df_pcpr['dt_decision'], format='%d.%m.%Y', errors='coerce')


In [136]:
# Merge on document_id to get common ids
common_ids = pd.merge(df_parp[['document_id']], df_pcpr[['document_id']], how='inner', on='document_id')

# Merge both DataFrames to get the dt_decision for these common ids
df_parp_common = df_parp[df_parp['document_id'].isin(common_ids['document_id'])]
df_pcpr_common = df_pcpr[df_pcpr['document_id'].isin(common_ids['document_id'])]
# "case_nature": document.get("case_nature"),
# "case_type": document.get("case_type"),
# "collector_name": document.get("collector_name"),
# Merge both DataFrames to compare dt_decision side-by-side
merged_common_diff = pd.merge(df_parp_common[['document_id', 'dt_decision', 'document_procedure', 'nature', "type", "coll_nom"]], 
                              df_pcpr_common[['document_id', 'dt_decision', 'document_procedure', 'nature', "type", "coll_nom"]], 
                              on='document_id', 
                              suffixes=('_parp', '_pcpr'))

# Filter where the dates are different
diff_dates_df = merged_common_diff[merged_common_diff['dt_decision_parp'] != merged_common_diff['dt_decision_pcpr']]

print(f"Records with same document_id but different dt_decision and procedure type: {len(diff_dates_df)}")
diff_dates_df


Records with same document_id but different dt_decision and procedure type: 147


,document_id,dt_decision_parp,document_procedure_parp,nature_parp,type_parp,coll_nom_parp,dt_decision_pcpr,document_procedure_pcpr,nature_pcpr,type_pcpr,coll_nom_pcpr
0,830,2013-06-05,PM/337/2013,EXE,None,parp,2012-09-18,P/2300/2011,TDP,None,pcpr
1,1576,2015-04-21,PM/308/2015,EXE,None,parp,2013-12-12,P/4847/2009,MP,None,pcpr
2,2020,2016-03-07,P/8517/2014,PENAL,None,parp,2014-09-23,P/9423/2014,MP,None,pcpr
3,2197,2016-10-26,P/4296/2012,PENAL,None,parp,2014-12-19,P/11665/2014,JMI,None,pcpr
4,None,NaT,None,None,None,None,NaT,None,None,None,None
...,...,...,...,...,...,...,...,...,...,...,...
142,3240,2018-12-20,P/593/2018,PENAL,None,parp,2016-03-08,P/24473/2015,MP,None,pcpr
143,3136,2019-02-01,P/7146/2018,PENAL,None,parp,2016-02-11,P/3871/2013,MP,None,pcpr
144,3251,2019-04-15,P/20201/2015,PENAL,None,parp,2016-03-31,P/24474/2014,MP,None,pcpr
145,3239,2019-03-20,P/12928/2015,CRIM,None,parp,2016-03-30,P/24473/2015,MP,None,pcpr


### CIVIL COURT

In [138]:
dataset = {
    "acjc": "Penal_Civil/decis_cjc_acjc.json", 
    "sommaires": "Penal_Civil/decis_cjc_sommaires.json"
}

In [139]:
for data, path in dataset.items():
    if data == "acjc":
        reader = DataReader(file_path=path)
        df_acjc = reader.retrieve_document_fields()
        print(reader.collect_keys())
    if data == "sommaires":
        reader = DataReader(file_path=path)
        df_sommaires = reader.retrieve_document_fields()
        print(reader.collect_keys())

{'source', 'n_ext_proc', 'coll_nom', 'datedecision', 'n_ext_jur_attr', 'document_text', 'cle_fiche', 'decision', 'relations', 'parties', 'resultat', 'recours', 'publieinternet', 'query', 'nature', 'resume', 'importance', 'fichierword', 'descripteurs', 'document', 'rectification', 'normes', 'dt_decision', 'arretdeprincipe', 'proc_year', 'id', 'procedure'}
{'source', 'n_ext_proc', 'coll_nom', 'datedecision', 'n_ext_jur_attr', 'document_text', 'cle_fiche', 'decision', 'relations', 'parties', 'resultat', 'recours', 'publieinternet', 'query', 'nature', 'resume', 'importance', 'fichierword', 'descripteurs', 'document', 'rectification', 'normes', 'dt_decision', 'proc_year', 'arretdeprincipe', 'id', 'procedure'}


In [140]:
df_acjc

,document_id,dt_decision,document_procedure,nature,type,coll_nom
0,1604572787,22.09.2020,C/26828/2017,OS,None,acjc
1,12278,23.05.2013,C/9537/2013,IUO,None,acjc
2,12281,30.08.2013,C/11578/2011,OSDF,None,acjc
3,12374,02.10.2013,C/6021/2012,SDF,None,acjc
4,12376,19.08.2013,C/25347/2012,SDF,None,acjc
...,...,...,...,...,...,...
7026,1689327529,16.06.2023,C/20476/2021,OS,None,acjc
7027,1725882074,09.09.2024,C/6683/2023,OO,None,acjc
7028,1681221742,28.03.2023,C/16675/2020,OO,None,acjc
7029,1725976647,30.08.2024,C/5939/2024,SDF,None,acjc


In [143]:
df_sommaires

,document_id,dt_decision,document_procedure,nature,type,coll_nom
0,1669808833,2022-11-17,C/24400/2021,SEX,None,sommaires
1,1716787799,2024-05-22,C/9/2023,SP,None,sommaires
2,1717052909,2024-05-28,C/751/2024,SFC,None,sommaires
3,1716542482,2024-05-23,C/5370/2024,SFC,None,sommaires
4,1717146487,2024-05-30,C/22038/2023,SML,None,sommaires
...,...,...,...,...,...,...
4177,1726135208,2024-09-09,C/12532/2024,SFC,None,sommaires
4178,1726137252,2024-09-09,C/12530/2024,SFC,None,sommaires
4179,1726139619,2024-09-10,C/9582/2024,SFC,None,sommaires
4180,1726123301,2024-09-10,C/696/2024,SFC,None,sommaires


In [144]:
# Convert dt_decision to datetime
df_acjc['dt_decision'] = pd.to_datetime(df_acjc['dt_decision'], format='%d.%m.%Y', errors='coerce')
df_sommaires['dt_decision'] = pd.to_datetime(df_sommaires['dt_decision'], format='%d.%m.%Y', errors='coerce')


In [147]:
# Merge on document_id to get common ids
common_ids = pd.merge(df_acjc[['document_id']], df_sommaires[['document_id']], how='inner', on='document_id')

# Merge both DataFrames to get the dt_decision for these common ids
df_acjc_common = df_acjc[df_acjc['document_id'].isin(common_ids['document_id'])]
df_sommaires_common = df_sommaires[df_sommaires['document_id'].isin(common_ids['document_id'])]

# Merge both DataFrames to compare dt_decision side-by-side
merged_common_diff = pd.merge(df_acjc_common[['document_id', 'dt_decision', 'document_procedure', 'nature', "type", "coll_nom"]], 
                              df_sommaires_common[['document_id', 'dt_decision', 'document_procedure', 'nature', "type", "coll_nom"]], 
                              on='document_id', 
                              suffixes=('_df_acjc', '_df_sommaires'))

# Filter where the dates are different
diff_dates_df = merged_common_diff[merged_common_diff['dt_decision_df_acjc'] != merged_common_diff['dt_decision_df_sommaires']]

print(f"Records with same document_id but different dt_decision and procedure type: {len(diff_dates_df)}")
diff_dates_df


Records with same document_id but different dt_decision and procedure type: 33


,document_id,dt_decision_df_acjc,document_procedure_df_acjc,nature_df_acjc,type_df_acjc,coll_nom_df_acjc,dt_decision_df_sommaires,document_procedure_df_sommaires,nature_df_sommaires,type_df_sommaires,coll_nom_df_sommaires
0,4961,1998-03-20,C/35317/1996,OO,None,acjc,2013-06-28,C/512/2013,SFC,None,sommaires
1,6954,2006-03-17,C/25198/2004,OO,None,acjc,2016-07-07,C/4364/2016,SFC,None,sommaires
2,6963,2006-03-17,C/18936/1999,OO,None,acjc,2016-07-13,C/26593/2015,SP,None,sommaires
3,7728,2007-06-08,C/3279/2004,OO,None,acjc,2017-09-20,C/26364/2016,SML,None,sommaires
4,7614,2007-03-16,C/19409/2005,OO,None,acjc,2017-08-15,C/7941/2017,SFC,None,sommaires
...,...,...,...,...,...,...,...,...,...,...,...
28,9088,2009-04-24,C/27267/2007,I,None,acjc,2019-05-06,C/24746/2017,SML,None,sommaires
29,None,NaT,None,None,None,None,NaT,None,None,None,None
30,8726,2008-11-14,C/6579/2007,OO,None,acjc,2018-12-19,C/23310/2017,SFC,None,sommaires
31,1713779580,2024-04-22,C/17382/2023,SDF,None,acjc,2024-04-18,C/22866/2023,SP,None,sommaires


In [161]:
dataset = {
    "adm": "Administrative_Court/output.jsonl",
}

In [162]:
for data, path in dataset.items():
    if data == "adm":
        reader = DataReader(file_path=path)
        df_admin = reader.retrieve_document_fields()
        print(reader.collect_keys())


{'source', 'n_ext_proc', 'coll_nom', 'datedecision', 'n_ext_jur_attr', 'document_text', 'decision', 'cle_fiche', 'relations', 'parties', 'resultat', 'recours', 'publieinternet', 'query', 'nature', 'resume', 'importance', 'fichierword', 'descripteurs', 'document', 'rectification', 'normes', 'dt_decision', 'proc_year', 'arretdeprincipe', 'id', 'procedure'}


In [163]:
df_admin

,document_id,dt_decision,document_procedure,nature,type,coll_nom
0,1685021361,16.05.2023,A/3741/2022,LIPAD,None,ata
1,1684917228,16.05.2023,A/3018/2022,PE,None,ata
2,1684995776,22.05.2023,A/491/2023,PRISON,None,ata
3,1685007976,23.05.2023,A/4261/2021,PE,None,ata
4,1684249336,16.05.2023,A/1327/2023,MC,None,ata
...,...,...,...,...,...,...
22291,13297,08.09.2009,A/940/2009,LDTR,None,ata
22292,1689834624,18.07.2023,A/3715/2022,EXPLOI,None,ata
22293,1703245869,12.12.2023,A/2120/2022,LDTR,None,ata
22294,1702656873,12.12.2023,A/115/2021,FPUBL,None,ata


### Compare the document_text fields for specific IDs (like 830 and 1576) between df_parp and df_pcpr

### Steps:
- Filter the rows with the matching document_id (830 and 1576).
- Compare the document_text fields for these IDs.
- Extract and highlight the differences between the texts

In [34]:
# Convert document_id to string in both DataFrames for consistency
df_parp['document_id'] = df_parp['document_id'].astype(str)
df_pcpr['document_id'] = df_pcpr['document_id'].astype(str)


In [38]:
from difflib import Differ

ids_to_compare = ['830', '1576']

df_parp_filtered = df_parp[df_parp['document_id'].isin(ids_to_compare)]
df_pcpr_filtered = df_pcpr[df_pcpr['document_id'].isin(ids_to_compare)]

def compare_texts(text1, text2):
    differ = Differ()
    diff = list(differ.compare(text1.splitlines(), text2.splitlines()))
    
    # Only return the differences (+ and - represent different parts)
    return '\n'.join([line for line in diff if line.startswith('+ ') or line.startswith('- ')])

for doc_id in ids_to_compare:

    # Check if the document_id is present in both DataFrames
    parp_row = df_parp_filtered[df_parp_filtered['document_id'] == doc_id]
    pcpr_row = df_pcpr_filtered[df_pcpr_filtered['document_id'] == doc_id]
    
    if parp_row.empty:
        print(f"Document ID {doc_id} not found in df_parp.")
        continue
    if pcpr_row.empty:
        print(f"Document ID {doc_id} not found in df_pcpr.")
        continue
    
    parp_text = parp_row['document_text'].values[0]
    pcpr_text = pcpr_row['document_text'].values[0]
    
    print(f"Differences for document_id {doc_id}:")
    differences = compare_texts(parp_text, pcpr_text)
    # print(differences)
    print("\n" + "="*50 + "\n")


Differences for document_id 830:


Differences for document_id 1576:




### Ensure that there are no empty or null values in the cle_fiche column of your DataFrame before checking the number of unique value

#### Explanation:
1. **`notna()`**: This checks for non-null values.
2. **`(df_pcpr['cle_fiche'] != '')`**: This ensures there are no empty strings.
3. **`unique()`**: This gets unique values from the filtered column.
4. **`len()`**: This counts the number of unique values.


In [39]:
# Filter out null or empty values in the 'cle_fiche' column
filtered_df_pcpr = df_pcpr[df_pcpr['cle_fiche'].notna() & (df_pcpr['cle_fiche'] != '')]
filtered_df_parp = df_parp[df_parp['cle_fiche'].notna() & (df_parp['cle_fiche'] != '')]


# Get the count of unique values in 'cle_fiche'
df_pcpr_unique_count = len(filtered_df_pcpr['cle_fiche'].unique())
df_parp_unique_count = len(filtered_df_parp['cle_fiche'].unique())

print(f"Number of unique non-null, non-empty 'cle_fiche' values for pcpr: {df_pcpr_unique_count}")
print(f"Number of unique non-null, non-empty 'cle_fiche' values for parp: {df_parp_unique_count}")


Number of unique non-null, non-empty 'cle_fiche' values for pcpr: 4980
Number of unique non-null, non-empty 'cle_fiche' values for parp: 3401


In [40]:
# Filter out null or empty values in the 'cle_fiche' column
filtered_df_acjc = df_acjc[df_acjc['cle_fiche'].notna() & (df_acjc['cle_fiche'] != '')]
filtered_df_sommaires = df_sommaires[df_sommaires['cle_fiche'].notna() & (df_sommaires['cle_fiche'] != '')]


# Get the count of unique values in 'cle_fiche'
df_acjc_unique_count = len(filtered_df_acjc['cle_fiche'].unique())
df_sommaires_unique_count = len(filtered_df_sommaires['cle_fiche'].unique())

print(f"Number of unique non-null, non-empty 'cle_fiche' values for acjc: {df_acjc_unique_count}")
print(f"Number of unique non-null, non-empty 'cle_fiche' values for sommaires: {df_sommaires_unique_count}")


Number of unique non-null, non-empty 'cle_fiche' values for acjc: 7030
Number of unique non-null, non-empty 'cle_fiche' values for sommaires: 4181


In [41]:
# ssl_context = ssl.create_default_context(cafile=CA_CERT_PATH)

es = Elasticsearch(
    ["http://localhost:9200"],
    basic_auth=("elastic", "cGV03C1p_20HEdwhZmFcBcoeQ845PT5tlZFV0YWI2zE"),
    verify_certs=False,
)

es

<Elasticsearch(['http://localhost:9200'])>

In [43]:
ES_JUDICIARY_INDEX_NAME = "legal_civil_loader6_new"

In [45]:
es.cluster.health()

ObjectApiResponse({'cluster_name': 'docker-cluster', 'status': 'yellow', 'timed_out': False, 'number_of_nodes': 1, 'number_of_data_nodes': 1, 'active_primary_shards': 21, 'active_shards': 21, 'relocating_shards': 0, 'initializing_shards': 0, 'unassigned_shards': 19, 'delayed_unassigned_shards': 0, 'number_of_pending_tasks': 0, 'number_of_in_flight_fetch': 0, 'task_max_waiting_in_queue_millis': 0, 'active_shards_percent_as_number': 52.5})

---

### Display the field types and definitions for the index.

In [48]:
# Fetch the mapping for the specified index
mapping = es.indices.get_mapping(index=ES_JUDICIARY_INDEX_NAME)

# Extract the properties (fields) from the mapping
fields = mapping[ES_JUDICIARY_INDEX_NAME]['mappings']['properties']

# Print all fields in the index
print("Indexed Fields:")
for field in fields: 
    print(field)


Indexed Fields:
appeals
collector_name
correction
decision_date
decision_type
descriptors
document
document_text
external_jurisdiction_attribute
external_procedure_number
id
importance
nature
parties_involved
procedure_type
procedure_year
published_internet
record_key
relations
result
source
standards
stopped_from_begin
summary
word_file


In [123]:
# Get the mapping of the Elasticsearch index
mapping = es.indices.get_mapping(index="civil_court_chamber10")

# Convert ObjectApiResponse to dictionary
mapping_dict = mapping.body  # `.body` returns the response as a dictionary

# Pretty-print the dictionary to JSON format
print(json.dumps(mapping_dict, indent=2))


{
  "civil_court_chamber10": {
    "mappings": {
      "properties": {
        "appeals": {
          "properties": {
            "action_key": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": "keyword",
                  "ignore_above": 256
                }
              }
            },
            "action_status": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": "keyword",
                  "ignore_above": 256
                }
              }
            },
            "appeal_type": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": "keyword",
                  "ignore_above": 256
                }
              }
            },
            "atf_number": {
              "type": "text",
              "fields": {
                "keyword": {
                  "type": "keyword",
                 

---

### Define an aggregation query to retrieve unique values for "appeals.chamber"


In [122]:
query = {
    "size": 0,  # Don't retrieve the documents themselves
    "aggs": {
        "unique_chambers": {
            "terms": {
                "field": "appeals.chamber.keyword",  # Use the keyword sub-field to get exact matches
                "size": 1000  # Set a sufficiently large size to capture all unique values
            }
        }
    }
}

# Execute the aggregation query
response = es.search(
    index="civil_court_chamber10",
    body=query,
)

# Retrieve and print the unique chambers from the aggregation results
unique_chambers = response["aggregations"]["unique_chambers"]["buckets"]
for chamber in unique_chambers:
    print(f"Chamber: {chamber['key']}, Count: {chamber['doc_count']}")


Chamber: 1, Count: 1144
Chamber: S1, Count: 382


### Verify new attribute named decision_chambre

In [165]:
query = {
    "size": 0,  # Don't retrieve the documents themselves
    "aggs": {
        "unique_chambers": {
            "terms": {
                "field": "decision_chamber.keyword",  # Use the keyword sub-field to get exact matches
                "size": 1000  # Set a sufficiently large size to capture all unique values
            }
        }
    }
}

# Execute the aggregation query
response = es.search(
    index="administrative_court",
    body=query,
)

# Retrieve and print the unique chambers from the aggregation results
unique_chambers = response["aggregations"]["unique_chambers"]["buckets"]
for chamber in unique_chambers:
    print(f"Chamber: {chamber['key']}, Count: {chamber['doc_count']}")


Chamber: Unknown, Count: 22295


In [166]:
query = {
    "size": 0,  # Don't retrieve the documents themselves
    "aggs": {
        "unique_chambers": {
            "terms": {
                "field": "collector_name.keyword",  # Use the keyword sub-field to get exact matches
                "size": 1000  # Set a sufficiently large size to capture all unique values
            }
        }
    }
}

# Execute the aggregation query
response = es.search(
    index="administrative_court",
    body=query,
)

# Retrieve and print the unique chambers from the aggregation results
unique_chambers = response["aggregations"]["unique_chambers"]["buckets"]
for chamber in unique_chambers:
    print(f"Chamber: {chamber['key']}, Count: {chamber['doc_count']}")


Chamber: ata, Count: 22295
